1. Defining the Predictive Problem

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

The business in question would like to predict the determining factors of Customer Churn. They seek this knowledge in order to reduce customer churn, retain clients and ultimately increase revenue generated from their current customer base.

Churn is important because it tells us how many clients are leaving and why. This allows the business to strategise against churn and retain their business.

In this case our target variable is churn - Yes/no. Whether or not the client will churn.


2. Understanding The Customer Data

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/Tyresej00/telco-dataset/refs/heads/main/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [ ]:
df.head(5)


In [ ]:
df.shape

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Churn Distribution by Contract")
plt.xlabel("Contract")
plt.ylabel("Count of Customers")
plt.show()

Month-month contracts have a signifigantly higher numbers of churn when compared to the longer contracts

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Churn Distribution by Monthly Charges")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")

As presented by the boxplot, there seems to be a correlation between higher monthly charges and customer churn.

In [ ]:
plt.figure(figsize=(8,5))
sns.kdeplot(data=df, x="tenure", hue="Churn", fill=True, common_norm=False)
plt.title("Churn Distribution by Tenure")
plt.xlabel("Tenure")
plt.ylabel("Density")


As we can see, there may be a correlation between tenure and churn, showing that a lower tenure can lead to a higher customer churn probability.


3. Preparing and Cleaning the Data


Handling Missing Values and Categorical Variables

In [ ]:
df.isnull().sum()

In [ ]:
print(df.dtypes)

TotalCharges is currently an object, however these are numeric values. We will be changing this column data type to numeric so that calculations can be drawn on it.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [ ]:
print(df.isnull().sum())

In [ ]:
MissingTotalCharge = df[df["TotalCharges"].isnull()]
MissingTotalCharge

It's clear that we have missing values, however we should split the data before imputing otherwise this may lead to data leakage.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

We are going to first split the data and then we can impute any missing values with the median of the train set.

In [ ]:
#split the data and determining the target feature

x = df.drop("Churn", axis=1)
y = df["Churn"]

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

#calculating median from the train set

train_median = x_train["TotalCharges"].median()

#fill in the missing values of both sets

x_train["TotalCharges"] = x_train["TotalCharges"].fillna(train_median)
x_test["TotalCharges"] = x_test["TotalCharges"].fillna(train_median)

#testing if any null values remain

print(x_train["TotalCharges"].isnull().sum())
print(x_test["TotalCharges"].isnull().sum())


We no longer have missing values in our dataset. We can now proceed to encoding the data using one-hot encoding.

I realised while looking at the data that we kept the customer ID column. This column needs to be removed as it serves no purpose in the model due to it being a unique identifier.

In [ ]:
x_train.drop(columns=['customerID'], axis=1, inplace=True)
x_test.drop(columns=['customerID'], axis=1, inplace=True)

We can now encode our categorical columns

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder

In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

categorical_columns = ["gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod"]
numerical_columns = ["MonthlyCharges", "TotalCharges", "tenure"]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns),
        ('num', StandardScaler(), numerical_columns)
        ])
x_train_final = preprocessor.fit_transform(x_train)
x_test_final = preprocessor.transform(x_test)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)


4. Building the Logistic Regression Model


In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
log_reg = LogisticRegression()

log_reg.fit(x_train_final, y_train_encoded)

5. Understanding Probabilities and Predictions


In [ ]:
y_pred_proba = log_reg.predict_proba(x_test_final)[:, 1]

y_pred = log_reg.predict(x_test_final)

print(y_pred_proba)
print(y_pred)
y_pred_proba.shape

6. Evaluating Model Performance


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.metrics import f1_score

In [ ]:
#Confusion Matrix
cm = confusion_matrix(y_test_encoded, y_pred)
print("Confusion Matrix:\n", cm)

In [ ]:
#F-1 Score
f1 = f1_score(y_test_encoded, y_pred)

print(f"F1-Score: {f1:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
y_pred = log_reg.predict(x_test_final)

In [ ]:
cm = confusion_matrix(y_test_encoded, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
print(classification_report(y_test_encoded, y_pred))

In [ ]:
feature_names = preprocessor.get_feature_names_out()
coefficients = log_reg.coef_[0]

#DataFrame used to view coefficients clearly
coef_df = pd.DataFrame({'Variable': feature_names, 'Coefficient': coefficients})

#Sort by absolute value to see the most influential features
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values(by='Abs_Coefficient', ascending=False)

print(coef_df)